In [1]:
%pip install tensorflow==2.16.2
%pip install keras-tuner==1.4.7
%pip install numpy<2.0.0




[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [keras-tuner]

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
zsh:1: no such file or directory: 2.0.0
Note: you may need to restart the kernel to use updated packages.


In [3]:
import sys

# Increase recursion limit to prevent potential issues
sys.setrecursionlimit(100000)

In [4]:
# Step 2: Import necessary libraries
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.datasets import mnist
from tensorflow.keras.optimizers import Adam
import os
import warnings

# Suppress all Python warnings
warnings.filterwarnings('ignore')

# Set TensorFlow log level to suppress warnings and info messages
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # 0 = all logs, 1 = filter out INFO, 2 = filter out INFO and WARNING, 3 = ERROR only



In [5]:

# Step 3: Load and preprocess the MNIST dataset
(x_train, y_train), (x_val, y_val) = mnist.load_data()
x_train, x_val = x_train / 255.0, x_val / 255.0

print(f'Training data shape: {x_train.shape}')
print(f'Validation data shape: {x_val.shape}')

Training data shape: (60000, 28, 28)
Validation data shape: (10000, 28, 28)


In [6]:
# Define a model-building function 

def build_model(hp):
    model = Sequential([
        Flatten(input_shape=(28, 28)),
        Dense(units=hp.Int('units', min_value=32, max_value=512, step=32), activation='relu'),
        Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG')),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


Configuring the hyperparameter search

In [9]:
# Create a RandomSearch Tuner
tuner = kt.RandomSearch(
    build_model,                     # Função que constrói o modelo e define os hyperparameters
    objective='val_accuracy',        # Métrica que queremos maximizar: acurácia de validação
    max_trials=10,                   # Número máximo de combinações diferentes de hyperparameters a testar
    executions_per_trial=2,          # Cada combinação será treinada 2 vezes para reduzir variação
    directory='my_dir',              # Pasta onde os resultados do tuning serão armazenados
    project_name='intro_to_kt'       # Nome do subdiretório do projeto (organiza execuções)
)

# Display a summary of the search space 
tuner.search_space_summary()         # Mostra os hyperparameters que o tuner vai explorar

Search space summary
Default search space size: 2
units (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 512, 'step': 32, 'sampling': 'linear'}
learning_rate (Float)
{'default': 0.0001, 'conditions': [], 'min_value': 0.0001, 'max_value': 0.01, 'step': None, 'sampling': 'log'}


Running the hyperparameter search

1. tuner.search()
É o método principal do Keras Tuner.

In [11]:
# Run the hyperparameter search 
tuner.search(x_train, y_train, epochs=5, validation_data=(x_val, y_val)) 

# Display a summary of the results 
tuner.results_summary() 

Trial 10 Complete [00h 00m 49s]
val_accuracy: 0.9666500091552734

Best val_accuracy So Far: 0.9789499938488007
Total elapsed time: 00h 06m 30s
Results summary
Results in my_dir/intro_to_kt
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 08 summary
Hyperparameters:
units: 416
learning_rate: 0.002320127603619074
Score: 0.9789499938488007

Trial 04 summary
Hyperparameters:
units: 320
learning_rate: 0.0022451540259137105
Score: 0.9785000085830688

Trial 07 summary
Hyperparameters:
units: 320
learning_rate: 0.0018578460759471264
Score: 0.9783500134944916

Trial 01 summary
Hyperparameters:
units: 352
learning_rate: 0.0020977643583507096
Score: 0.9778999984264374

Trial 02 summary
Hyperparameters:
units: 320
learning_rate: 0.0028301052432755675
Score: 0.977400004863739

Trial 06 summary
Hyperparameters:
units: 128
learning_rate: 0.0021321979115285347
Score: 0.9763000011444092

Trial 05 summary
Hyperparameters:
units: 64
learning_rate: 0.000602358853272268
Score: 

Analyzing and using the best hyperparameters

In [12]:
# Step 1: Retrieve the best hyperparameters 

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0] 
print(f""" 

The optimal number of units in the first dense layer is {best_hps.get('units')}. 

The optimal learning rate for the optimizer is {best_hps.get('learning_rate')}. 

""") 

# Step 2: Build and Train the Model with Best Hyperparameters 
model = tuner.hypermodel.build(best_hps) 
model.fit(x_train, y_train, epochs=10, validation_split=0.2) 

# Evaluate the model on the test set 
test_loss, test_acc = model.evaluate(x_val, y_val) 
print(f'Test accuracy: {test_acc}') 

 

The optimal number of units in the first dense layer is 416. 

The optimal learning rate for the optimizer is 0.002320127603619074. 


Epoch 1/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9389 - loss: 0.2034 - val_accuracy: 0.9613 - val_loss: 0.1235
Epoch 2/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9732 - loss: 0.0871 - val_accuracy: 0.9678 - val_loss: 0.1050
Epoch 3/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9798 - loss: 0.0627 - val_accuracy: 0.9753 - val_loss: 0.0959
Epoch 4/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9850 - loss: 0.0471 - val_accuracy: 0.9706 - val_loss: 0.1160
Epoch 5/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9866 - loss: 0.0416 - val_accuracy: 0.9695 - val_loss: 0.1262
Epoch 6/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9887 - loss: 0.0332 - val_accuracy: 0.9738 - val_loss: 0.1075
Epoch 7/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9911 - loss: 0